# Attempt 1

In [0]:
!pip install youtube-transcript-api
!pip install langchain
!pip install langchain-community
!pip install langchain-experimental
!pip install sentence-transformers
!pip install faiss-cpu
!pip install ollama

In [0]:
# ============================================================
# YOUTUBE VIDEO RAG APPLICATION (FULLY FREE)
# ============================================================
#
# FEATURES:
# - Takes YouTube video URL
# - Extracts captions/transcript
# - Creates semantic chunks
# - Generates embeddings
# - Stores vectors in FAISS
# - Uses local LLM via Ollama
# - Answers user questions from video
#
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================
#
# pip install youtube-transcript-api
# pip install langchain
# pip install langchain-community
# pip install langchain-experimental
# pip install sentence-transformers
# pip install faiss-cpu
# pip install ollama
#
# Install Ollama:
# https://ollama.com
#
# Pull local model:
# ollama pull llama3
#
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs

from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.llms import Ollama
# from langchain_core.prompts import PromptTemplate
from langchain.chains.retrieval_qa.base import RetrievalQA

## Code

In [0]:



# ============================================================
# STEP 1 : EXTRACT VIDEO ID FROM URL
# ============================================================

def extract_video_id(youtube_url):

    parsed_url = urlparse(youtube_url)

    if parsed_url.hostname == "youtu.be":
        return parsed_url.path[1:]

    if parsed_url.hostname in (
        "www.youtube.com",
        "youtube.com",
        "m.youtube.com"
    ):
        return parse_qs(parsed_url.query)["v"][0]

    raise ValueError("Invalid YouTube URL")


# ============================================================
# STEP 2 : FETCH TRANSCRIPT
# ============================================================

def get_youtube_transcript(video_id):

    transcript = YouTubeTranscriptApi.get_transcript(video_id)

    full_text = " ".join([item["text"] for item in transcript])

    return full_text


# ============================================================
# STEP 3 : LOAD EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model...\n")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# ============================================================
# STEP 4 : CREATE SEMANTIC CHUNKS
# ============================================================

def create_chunks(text):

    print("\nCreating semantic chunks...\n")

    text_splitter = SemanticChunker(
        embeddings=embeddings
    )

    documents = text_splitter.create_documents([text])

    return documents


# ============================================================
# STEP 5 : CREATE VECTOR DATABASE
# ============================================================

def create_vector_store(documents):

    print("\nCreating FAISS vector store...\n")

    vectorstore = FAISS.from_documents(
        documents,
        embeddings
    )

    return vectorstore


# ============================================================
# STEP 6 : LOAD LOCAL LLM USING OLLAMA
# ============================================================

print("\nLoading local LLM...\n")

llm = Ollama(model="llama3")


# ============================================================
# STEP 7 : CUSTOM PROMPT TEMPLATE
# ============================================================

prompt_template = """
You are a helpful AI assistant.

Answer the question ONLY from the provided transcript context.

If the answer is not present in the context,
say:
"I could not find the answer in the video transcript."

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


# ============================================================
# STEP 8 : BUILD RAG PIPELINE
# ============================================================

def build_rag_chain(vectorstore):

    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={
            "prompt": PROMPT
        }
    )

    return qa_chain


# ============================================================
# STEP 9 : MAIN APPLICATION
# ============================================================

def main():

    print("\n==============================")
    print("YOUTUBE VIDEO RAG APPLICATION")
    print("==============================\n")

    youtube_url = input("Enter YouTube Video URL:\n")

    try:

        # ------------------------------------
        # Extract Video ID
        # ------------------------------------
        video_id = extract_video_id(youtube_url)

        print(f"\nVideo ID: {video_id}")

        # ------------------------------------
        # Fetch Transcript
        # ------------------------------------
        print("\nFetching transcript...\n")

        transcript_text = get_youtube_transcript(video_id)

        print("\nTranscript fetched successfully!")

        # ------------------------------------
        # Chunking
        # ------------------------------------
        documents = create_chunks(transcript_text)

        print(f"\nTotal chunks created: {len(documents)}")

        # ------------------------------------
        # Create Vector Store
        # ------------------------------------
        vectorstore = create_vector_store(documents)

        # ------------------------------------
        # Build QA Chain
        # ------------------------------------
        qa_chain = build_rag_chain(vectorstore)

        print("\nRAG Pipeline Ready!")

        # ------------------------------------
        # Ask Questions Loop
        # ------------------------------------
        while True:

            print("\n----------------------------------")

            query = input("\nAsk Question (or type exit):\n")

            if query.lower() == "exit":
                print("\nExiting application...")
                break

            print("\nGenerating Answer...\n")

            response = qa_chain.run(query)

            print("\nANSWER:\n")
            print(response)

    except Exception as e:

        print("\nERROR OCCURRED:\n")
        print(str(e))


# ============================================================
# RUN APPLICATION
# ============================================================

if __name__ == "__main__":
    main()

# Attempt 2

In [0]:
# Document Loader
from youtube_transcript_api import YouTubeTranscriptApi

yt_api = YouTubeTranscriptApi()
video_id = 'zoqKwqM7wkQ'
transcripts = yt_api.fetch(video_id)

In [0]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "zoqKwqM7wkQ"

try:
    transcript = YouTubeTranscriptApi.get_transcript(video_id)

    full_text = " ".join([x["text"] for x in transcript])

    print(full_text)

except Exception as e:
    print(e)

In [0]:
from langchain.text_splitter import RecursiveCharacterTextSplitter